# Redução de Dimensionalidade das Métricas Locais via PCA

In [2]:
import pandas as pd
import numpy as np
from scipy.stats import ttest_ind


from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.impute import SimpleImputer
from scipy.stats import mannwhitneyu
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score

from scipy.stats import ttest_ind

import plotly.express as px
import plotly.io as pio

pio.templates.default = "plotly_dark"


### Carregando o dataset

In [5]:
DATASET_PATH = "../../data/processed/dataset_features.parquet"

df_dataset = pd.read_parquet(DATASET_PATH)
df_dataset.shape
df_filtered = df_dataset.columns[df_dataset.columns.str.contains("age", case=False)]
df_filtered



Index(['age', 'age_25-30', 'age_30-35', 'age_55-60', 'age_60-65', 'age_65-70',
       'age_70-75', 'age_75-80'],
      dtype='object')

In [6]:
df_dataset.head()

,degree_0,degree_1,degree_2,degree_3,degree_4,degree_5,degree_6,degree_7,degree_8,degree_9,...,age_60-65,age_65-70,age_70-75,age_75-80,gender_2,handedness_left,handedness_right,education_gymnasium,education_realschule,relationship_status_yes
0,14786.0,14223.0,13305.0,12562.0,9459.0,13334.0,11893.0,8054.0,8195.0,15173.0,...,False,False,False,False,True,False,True,True,False,False
1,20374.0,20232.0,24459.0,20399.0,15591.0,16955.0,16395.0,15200.0,13285.0,22416.0,...,False,False,False,False,True,False,True,True,False,False
2,22982.0,24351.0,22231.0,17394.0,14843.0,20348.0,21595.0,21675.0,12499.0,16804.0,...,False,False,False,False,True,False,True,False,True,True
3,8682.0,9045.0,16592.0,12541.0,8066.0,11202.0,15684.0,15513.0,8463.0,10332.0,...,False,False,False,False,True,False,True,False,True,True
4,29168.0,28481.0,26967.0,27563.0,27400.0,24913.0,23154.0,24220.0,20045.0,14112.0,...,False,False,False,False,False,True,False,True,False,False


### Seleção Métricas Locais

In [7]:
NODE_FEATURE_PREFIXES = [
    "degree_",
    "clustering_",
    "betweenness_",
    "local_efficiency_",
    "hub_frequency_",
    "weighted_degree_"
]

NODE_FEATURES = [
    c for c in df_dataset.columns
    if any(c.startswith(p) for p in NODE_FEATURE_PREFIXES)
]

NODE_GROUPS = {
    "degree": [c for c in df_dataset.columns if c.startswith("degree_")],
    "clustering": [c for c in df_dataset.columns if c.startswith("clustering_")],
    "betweenness": [c for c in df_dataset.columns if c.startswith("betweenness_")],
    "local_efficiency": [c for c in df_dataset.columns if c.startswith("local_efficiency_")],
    "weighted_degree": [c for c in df_dataset.columns if c.startswith("weighted_degree_")]
}


len(NODE_FEATURES)


366

In [8]:
X = df_dataset[NODE_FEATURES].values
y = df_dataset["condition"].values

### Padronização das Feaures

Substituindo Nan por zero:

In [9]:
X = np.nan_to_num(X, nan=0.0)

In [10]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)


In [11]:
X_scaled.mean(axis=0)[:5], X_scaled.std(axis=0)[:5]

(array([-1.60365548e-16,  2.28212511e-16,  1.06910365e-16, -6.78469626e-17,
         1.31581988e-16]),
 array([1., 1., 1., 1., 1.]))

### PCA exploratório

In [12]:
np.isnan(X).sum()


np.int64(0)

In [13]:
nan_per_feature = pd.Series(np.isnan(X).sum(axis=0), index=NODE_FEATURES)
nan_per_feature[nan_per_feature > 0].sort_values(ascending=False)


Series([], dtype: int64)

In [14]:
pca_full = PCA()
X_pca_full = pca_full.fit_transform(X_scaled)

explained_var = pca_full.explained_variance_ratio_
cum_var = np.cumsum(explained_var)


In [15]:
df_var = pd.DataFrame({
    "component": np.arange(1, len(explained_var) + 1),
    "explained_variance": explained_var,
    "cumulative_variance": cum_var
})


In [16]:
df_var

,component,explained_variance,cumulative_variance
0,1,4.372908e-01,0.437291
1,2,5.859918e-02,0.495890
2,3,5.130863e-02,0.547199
3,4,2.793894e-02,0.575137
4,5,2.454108e-02,0.599679
...,...,...,...
103,104,1.981148e-04,0.999530
104,105,1.792387e-04,0.999709
105,106,1.653188e-04,0.999875
106,107,1.252523e-04,1.000000


In [17]:
fig = px.line(
    df_var,
    x="component",
    y="cumulative_variance",
    markers=True,
    title="Variância Acumulada Explicada pelo PCA"
)
fig.add_hline(y=0.8, line_dash="dash", annotation_text="80%")
fig.add_hline(y=0.9, line_dash="dash", annotation_text="90%")
fig.show()


Observa-se que aproximadamente 14 componentes principais são suficientes para explicar cerca de 80% da variância total dos dados, enquanto 29 componentes explicam cerca de 90%, indicando elevada redundância entre as métricas de rede.

### PCA com 2 componentes

In [18]:
NODE_FEATURE_PREFIXES = [
    "degree_",
    "clustering_",
    "betweenness_",
    "local_efficiency_",
    "hub_frequency_",
    "weighted_degree_"
]

# Create a dictionary where keys are the metric names (e.g., 'degree')
NODE_FEATURES_BY_TYPE = {
    prefix.rstrip("_"): [c for c in NODE_FEATURES if c.startswith(prefix)]
    for prefix in NODE_FEATURE_PREFIXES
}



In [19]:
pca_2 = PCA(n_components=2)
X_pca_2 = pca_2.fit_transform(X_scaled)

df_pca = pd.DataFrame(
    X_pca_2,
    columns=["PC1", "PC2"]
)
df_pca["condition"] = y


In [20]:
fig = px.scatter(
    df_pca,
    x="PC1",
    y="PC2",
    color="condition",
    title="Projeção PCA das Métricas Locais (2 Componentes)",
    opacity=0.8
)
fig.show()


Aqui você responde visualmente:

existe separação EO × EC? Não há separação clara entre EO e EC

existe sobreposição? Sim

clusters? Existem regiões de maior densidade, mas não clusters bem definidos.

A projeção das métricas locais no espaço definido pelos dois primeiros
componentes principais não evidenciou uma separação clara entre as
condições EO e EC. Observa-se uma ampla sobreposição entre os pontos
correspondentes às duas condições experimentais, indicando que os
padrões dominantes de variabilidade capturados pelos componentes
principais são compartilhados entre os grupos.

Embora seja possível identificar regiões de maior densidade de pontos,
não foram observados clusters bem definidos ou fronteiras visuais que
permitam distinguir claramente as condições experimentais no espaço
PC1–PC2.


### Contribuição das features

In [21]:
loadings = pd.DataFrame(
    pca_2.components_.T,
    index=NODE_FEATURES,
    columns=["PC1", "PC2"]
)


In [22]:
loadings

,PC1,PC2
degree_0,0.071535,-0.054782
degree_1,0.069813,-0.044938
degree_2,0.074028,-0.033816
degree_3,0.071386,-0.053570
degree_4,0.068802,-0.072975
...,...,...
weighted_degree_56,0.069625,0.043515
weighted_degree_57,0.063101,0.074551
weighted_degree_58,0.046313,0.134602
weighted_degree_59,0.036310,0.157381


In [23]:
loadings.abs().sort_values("PC1", ascending=False).head(10)

,PC1,PC2
clustering_22,0.083809,0.006700
clustering_52,0.083548,0.003325
clustering_27,0.083402,0.014691
clustering_21,0.083111,0.013145
clustering_16,0.083069,0.010215
clustering_53,0.082550,0.002831
clustering_55,0.082540,0.002011
clustering_24,0.082359,0.014489
clustering_25,0.081760,0.010142
clustering_17,0.081728,0.000674


In [24]:
loadings.abs().sort_values("PC2", ascending=False).head(10)

,PC1,PC2
local_efficiency_59,0.014245,0.162572
hub_frequency_29,0.031629,0.162474
weighted_degree_59,0.036310,0.157381
degree_59,0.036310,0.157381
clustering_59,0.042188,0.145316
hub_frequency_28,0.029613,0.143491
local_efficiency_58,0.014315,0.139998
hub_frequency_18,0.008061,0.135185
degree_58,0.046313,0.134602
weighted_degree_58,0.046313,0.134602


Quais métricas e nós mais contribuem para cada componente?

PC1 separa redes mais ou menos conectadas globalmente, enquanto PC2 captura diferenças na organização local e eficiência estrutural.

### PCA com mais componentes - Para ML

In [25]:
pca_n = PCA(n_components=0.9)  # mantém 90% da variância
X_pca_n = pca_n.fit_transform(X_scaled)

X_pca_n.shape

(108, 40)

Para as análises de aprendizado de máquina, foi utilizado PCA com retenção de 90% da variância, resultando em 29 componentes principais.

A aplicação do PCA sobre as métricas locais permitiu reduzir significativamente
a dimensionalidade do conjunto de dados, preservando a maior parte da variância
original.

A projeção bidimensional indica padrões de organização distintos entre as
condições experimentais, embora com sobreposição parcial, sugerindo que métodos
não lineares ou classificadores supervisionados podem capturar diferenças mais sutis.

Os loadings indicam que métricas associadas a determinados nós apresentam maior
contribuição para os componentes principais, reforçando a relevância da análise
local da conectividade funcional.


-----

### Teste Estatístico

In [26]:
pc1_eo = df_pca.loc[df_pca.condition == "EO", "PC1"]
pc1_ec = df_pca.loc[df_pca.condition == "EC", "PC1"]

ttest_ind(pc1_eo, pc1_ec)

TtestResult(statistic=np.float64(0.5610825635664549), pvalue=np.float64(0.5759254603611156), df=np.float64(106.0))

In [27]:

mannwhitneyu(pc1_eo, pc1_ec, alternative="two-sided")


MannwhitneyuResult(statistic=np.float64(1576.0), pvalue=np.float64(0.4703105416093516))

In [28]:
px.box(df_pca, x="condition", y="PC1")


In [29]:
px.box(df_pca, x="condition", y="PC2")


In [30]:
df_pca = df_pca[df_pca["condition"].isin(["EO", "EC"])]

In [31]:


X = df_pca[["PC1", "PC2"]]
y = df_pca["condition"]

model = LogisticRegression()
scores = cross_val_score(model, X, y, cv=5)

scores.mean()


np.float64(0.5277056277056277)

As duas primeiras componentes principais não apresentaram capacidade discriminativa entre as condições EO e EC, uma vez que a acurácia média obtida foi próxima ao nível de acaso (≈50%).

In [37]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score
from sklearn.pipeline import Pipeline

X = df_dataset[NODE_FEATURES]
y = df_dataset["condition"]

model = Pipeline(
    [
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        (
            "clf",
            LogisticRegression(
                solver="saga",
                l1_ratio=1.0,  # equivalente a L1 (penalty='l1' deprecated)
                max_iter=5000,
            ),
        ),
    ]
)


In [ ]:
scores = cross_val_score(model, X, y, cv=5)
scores.mean()


np.float64(0.4913419913419913)

: 

Embora o primeiro e o segundo componentes principais sintetizem parte importante da variabilidade das métricas de rede, não foi observada separação visual ou estatisticamente significativa entre os grupos EC e EO no plano bidimensional. A análise inferencial do PC1 não indicou diferença entre os grupos (teste t: p = 0,575; Mann–Whitney: p = 0,470). De modo consistente, a regressão logística baseada em PC1 e PC2 apresentou acurácia próxima ao acaso, indicando que essa projeção bidimensional não discrimina adequadamente as condições.